# 财务数据批量同步（手动调度）

**用途**：前端、后端、数据库全部零改动，通过本 notebook 手动调取 ams_trae 后端**现有单只同步接口**，按 symbol 列表批量更新财务 / 估值 / 市值数据。

**关键设计**：每一次 HTTP 请求之后随机停顿 1–3 秒（而非每个 symbol 只停一次），避免 gm 接口锁定——这样同一 symbol 的多张表之间也会留出间隔。

**前置条件**：
1. ams_trae 后端已启动：`conda run -n fast311 uvicorn server_fast.main:app --host 0.0.0.0 --port 8008`
2. gm 终端已登录启动（否则同步返回 `error` / `no_data`）

**运行顺序**：配置 → 选择 symbol 来源 → 执行批量同步 → 结果汇总。失败项用末尾的 Cell 重试。


## 接口与口径约定

- **接口**：`POST /api/v1/bds/sync/{table}?symbol=xxx`（单只单表，后端现有接口，零改动）。
- **表名 `table`**：`fund-balance` / `fund-income` / `fund-cashflow` / `finance-deriv` / `daily-valuation` / `daily-mktvalue`。
- **返回**：HTTP 200，body `status` 三态 —— `success`（有新数据）/ `no_data`（无新数据，或 gm 未启动）/ `error`（失败）。**必须按 body.status 判断**，不能只看 HTTP 状态码。
- **增量**：财报按 `rpt_date` 增量（带 pub_date 保留策略）；每日估值 / 市值按 `trade_date` 增量。**重复运行幂等安全**。
- **首次全量**：库中无该 symbol 数据时从 2010-01-01 全量拉取，单只耗时会明显变长，超时已设 60s。
- **断点续跑**：已完成的 `(symbol, table)` 记录在 `sync_checkpoint.json`，重跑自动跳过。


In [5]:
# ===================== 配置 =====================
import os
import json
import time
import random

import requests
import pandas as pd

# 后端地址（ams_trae server_fast 默认 8008）
API_BASE = 'http://127.0.0.1:8008/api/v1'

# 要同步的表：4 张财报 + 每日估值 / 市值（默认全跑）
TABLES = [
    'fund-balance',     # 资产负债表
    'fund-income',      # 利润表
    'fund-cashflow',    # 现金流量表
    'finance-deriv',    # 财务指标
    'daily-valuation',  # 每日估值
    'daily-mktvalue',   # 每日市值
]

# 停顿：每次 HTTP 请求后随机停顿（秒），避免 gm 接口锁定
SLEEP_MIN, SLEEP_MAX = 1.0, 3.0

# 超时与重试
REQUEST_TIMEOUT = 60      # 单只单表超时（首次全量可能较慢）
MAX_RETRY = 2             # 单只单表失败重试次数
RETRY_BACKOFF = 5.0       # 重试间隔基数（第 n 次重试前停 RETRY_BACKOFF*n 秒）

# 连续失败退避：连续失败达到阈值后额外停顿一次
FAIL_STREAK_LIMIT = 5     # 连续失败次数阈值
FAIL_STREAK_SLEEP = 15.0  # 额外停顿秒数

# 断点续跑与结果落盘
CHECKPOINT_FILE = 'sync_checkpoint.json'
RESULT_CSV = 'sync_results.csv'

# symbol 前缀白名单（过滤非股票标的）
MARKET_PREFIXES = ('SHSE.', 'SZSE.', 'BJSE.')

SESSION = requests.Session()
print('配置就绪 | 后端:', API_BASE)
print('待同步表:', TABLES)


配置就绪 | 后端: http://127.0.0.1:8008/api/v1
待同步表: ['fund-balance', 'fund-income', 'fund-cashflow', 'finance-deriv', 'daily-valuation', 'daily-mktvalue']


In [6]:
# ===================== 工具函数 =====================

def _api_get(path, **params):
    '''GET 后端接口并返回 JSON。'''
    r = SESSION.get(f'{API_BASE}{path}', params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def _fetch_paged(path, page_size=1000, **params):
    '''分页拉取列表接口，返回全部 items（自动翻页直到取完）。'''
    items, offset = [], 0
    while True:
        j = _api_get(path, limit=page_size, offset=offset, **params)
        batch = j.get('items', [])
        items.extend(batch)
        total = j.get('total', 0)
        offset += page_size
        if not batch or offset >= total:
            break
    return items


def _filter_market(symbols):
    '''仅保留白名单前缀的股票标的，去重并保持顺序。'''
    seen, out = set(), []
    for s in symbols:
        if s and s.startswith(MARKET_PREFIXES) and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def resolve_symbols(scope, explicit=None, industry='', index_code='', max_symbols=0):
    '''按 scope 解析 symbol 列表。

    scope: explicit | all | industry | index
    '''
    scope = scope.lower()
    if scope == 'explicit':
        raw = list(explicit or [])
        note = '手写清单'
    elif scope == 'all':
        raw = [it['symbol'] for it in _fetch_paged('/bds/symbol-infos')]
        note = '全市场（bds_symbol_info）'
    elif scope == 'industry':
        if not industry:
            raise ValueError('scope=industry 需填 SCOPE_INDUSTRY')
        raw = [it['symbol'] for it in _fetch_paged('/bds/symbol-infos', industry=industry)]
        note = f'行业={industry}'
    elif scope == 'index':
        if not index_code:
            raise ValueError('scope=index 需填 SCOPE_INDEX_CODE')
        head = _api_get('/bds/index-constituents', index_code=index_code, limit=1, offset=0)
        items = head.get('items', [])
        if not items:
            raw, note = [], f'指数={index_code}（无成分数据）'
        else:
            td = items[0]['trade_date']
            rows = _fetch_paged('/bds/index-constituents', index_code=index_code, trade_date=td)
            raw = [it['symbol'] for it in rows]
            note = f'指数={index_code} @ {td}'
    else:
        raise ValueError(f'未知 scope: {scope}')

    syms = _filter_market(raw)
    if max_symbols and max_symbols > 0:
        syms = syms[:max_symbols]
    print(f'symbol 来源: {note} | 原始 {len(raw)} -> 过滤后 {len(syms)}')
    return syms


def sync_one(symbol, table):
    '''调用单只同步接口，返回 {status, message}。

    status: success | no_data | error
    '''
    url = f'{API_BASE}/bds/sync/{table}'
    last_err = ''
    for attempt in range(1, MAX_RETRY + 1):
        try:
            r = SESSION.post(url, params={'symbol': symbol}, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                body = r.json()
                st = body.get('status')
                msg = body.get('message', '')
                if st in ('success', 'no_data'):
                    return {'status': st, 'message': msg}
                last_err = msg or 'sync error'  # status == error
            else:
                # HTTPException：body 为 {'detail': ...}
                try:
                    last_err = f'HTTP {r.status_code}: ' + str(r.json().get('detail', r.text[:200]))
                except Exception:
                    last_err = f'HTTP {r.status_code}: {r.text[:200]}'
        except Exception as e:
            last_err = f'{type(e).__name__}: {e}'
        if attempt < MAX_RETRY:
            time.sleep(RETRY_BACKOFF * attempt)
    return {'status': 'error', 'message': last_err}


def load_checkpoint():
    '''读取已完成任务集合 {symbol|table}。'''
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                return set(json.load(f).get('done', []))
        except Exception:
            pass
    return set()


def save_checkpoint(done_set):
    '''写回已完成任务集合。'''
    try:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({'done': sorted(done_set)}, f, ensure_ascii=False)
    except Exception as e:
        print('checkpoint 写入失败:', e)


print('工具函数就绪')


工具函数就绪


In [7]:
# ===================== 选择 symbol 来源 =====================
# scope 取值：
#   'explicit' 手写清单（SYMBOLS_EXPLICIT）
#   'all'      全市场（bds_symbol_info 全部股票）
#   'industry' 指定行业（SCOPE_INDUSTRY，行业名可用 /bds/symbol-industries 查看）
#   'index'    指数成分（SCOPE_INDEX_CODE，如 SHSE.000300）
SCOPE = 'index'

SYMBOLS_EXPLICIT = [
    # 'SHSE.600519',   # 贵州茅台
    # 'SZSE.000001',   # 平安银行
]
SCOPE_INDUSTRY = ''        # scope=industry 时填，如 '银行'
SCOPE_INDEX_CODE = 'SHSE.000300'      # scope=index 时填，如 'SHSE.000300'
MAX_SYMBOLS = 0            # >0 时只取前 N 只（试跑用，0=不限）

SYMBOLS = resolve_symbols(
    SCOPE,
    explicit=SYMBOLS_EXPLICIT,
    industry=SCOPE_INDUSTRY,
    index_code=SCOPE_INDEX_CODE,
    max_symbols=MAX_SYMBOLS,
)
print('预览:', SYMBOLS[:10], '...' if len(SYMBOLS) > 10 else '')
print('总任务数:', len(SYMBOLS) * len(TABLES), '（symbol 数 x 表数）')


symbol 来源: 指数=SHSE.000300 @ 2026-10-12 | 原始 300 -> 过滤后 300
预览: ['SHSE.601377', 'SHSE.601901', 'SHSE.601688', 'SHSE.601788', 'SHSE.600809', 'SZSE.000768', 'SZSE.000776', 'SHSE.600893', 'SZSE.000792', 'SZSE.000807'] ...
总任务数: 1800 （symbol 数 x 表数）


In [8]:
# ===================== 执行批量同步 =====================
# RESUME=True：读取 checkpoint，跳过已完成 (symbol, table)
# RESET_CHECKPOINT=True：先清空 checkpoint 从头来（谨慎）
RESUME = True
RESET_CHECKPOINT = False


def run_tasks(tasks, resume=True, reset=False):
    '''执行 (symbol, table) 任务列表，返回结果记录列表。

    tasks: [(symbol, table), ...]
    '''
    global RESULTS
    if reset and os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
    done_set = load_checkpoint() if resume else set()

    todo = [(s, t) for (s, t) in tasks if f'{s}|{t}' not in done_set]
    skip = len(tasks) - len(todo)
    print(f'任务总数 {len(tasks)} | 跳过已完成 {skip} | 待执行 {len(todo)}')

    RESULTS = []
    total = len(todo)
    if total == 0:
        print('无待执行任务。')
        return RESULTS

    t0 = time.time()
    fail_streak = 0
    for i, (sym, table) in enumerate(todo, 1):
        res = sync_one(sym, table)
        RESULTS.append({'symbol': sym, 'table': table, **res})
        done_set.add(f'{sym}|{table}')
        save_checkpoint(done_set)

        if res['status'] in ('success', 'no_data'):
            fail_streak = 0
        else:
            fail_streak += 1

        pause = random.uniform(SLEEP_MIN, SLEEP_MAX)
        if fail_streak >= FAIL_STREAK_LIMIT:
            pause += FAIL_STREAK_SLEEP
            print(f'  连续失败 {fail_streak} 次，额外停顿 {FAIL_STREAK_SLEEP:.0f}s')
            fail_streak = 0
        time.sleep(pause)

        if i % 10 == 0 or i == total:
            elapsed = time.time() - t0
            eta = elapsed / i * (total - i)
            print(f'[{i}/{total}] {sym} {table:15s} -> {res["status"]:7s} | '
                  f'已用 {elapsed:.0f}s 预计剩余 {eta:.0f}s | {res["message"][:40]}')
    print(f'完成，用时 {time.time() - t0:.0f}s')
    return RESULTS


TASKS = [(s, t) for s in SYMBOLS for t in TABLES]
RESULTS = run_tasks(TASKS, resume=RESUME, reset=RESET_CHECKPOINT)


任务总数 1800 | 跳过已完成 0 | 待执行 1800
[10/1800] SHSE.601901 finance-deriv   -> success | 已用 52s 预计剩余 9390s | 同步完成：SHSE.601901，更新 71 条
[20/1800] SHSE.601788 fund-income     -> success | 已用 129s 预计剩余 11512s | 同步完成：SHSE.601788，更新 88 条
[30/1800] SHSE.600809 daily-mktvalue  -> success | 已用 237s 预计剩余 13994s | 同步完成：SHSE.600809，更新 7961 条
[40/1800] SZSE.000776 finance-deriv   -> success | 已用 313s 预计剩余 13777s | 同步完成：SZSE.000776，更新 111 条
[50/1800] SZSE.000792 fund-income     -> success | 已用 439s 预计剩余 15354s | 同步完成：SZSE.000792，更新 112 条
[60/1800] SZSE.000807 daily-mktvalue  -> success | 已用 566s 预计剩余 16416s | 同步完成：SZSE.000807，更新 6914 条
[70/1800] SHSE.601398 finance-deriv   -> success | 已用 620s 预计剩余 15321s | 同步完成：SHSE.601398，更新 85 条


KeyboardInterrupt: 

In [ ]:
# ===================== 结果汇总 =====================
df = pd.DataFrame(RESULTS)
print('本次执行条数:', len(df))
if not df.empty:
    print('\n状态分布：')
    print(df['status'].value_counts().to_string())

    print('\n各表 x 状态：')
    pivot = (df.pivot_table(index='table', columns='status', values='symbol', aggfunc='count')
               .fillna(0).astype(int))
    print(pivot.to_string())

    err = df[df['status'] == 'error']
    if not err.empty:
        print(f'\n失败明细（前 20，共 {len(err)}）：')
        print(err.head(20).to_string())

    df.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')
    print(f'\n结果已落盘: {RESULT_CSV}')


In [ ]:
# ===================== 仅重试失败项 =====================
# 重跑上一次执行中 status=error 的 (symbol, table)
RETRY_NO_DATA = True   # True: 连同 no_data（可能是 gm 未启动）一并重试

res_df = pd.DataFrame(RESULTS)
if RETRY_NO_DATA:
    failed = res_df[res_df['status'].isin(['error', 'no_data'])]
else:
    failed = res_df[res_df['status'] == 'error']
retry_tasks = list(dict.fromkeys(zip(failed['symbol'], failed['table'])))
print('待重试任务数:', len(retry_tasks))

if retry_tasks:
    df2 = pd.DataFrame(run_tasks(retry_tasks, resume=False, reset=False))
    RESULTS = RESULTS + df2.to_dict('records')
    print('重试后状态分布：')
    print(pd.DataFrame(RESULTS)['status'].value_counts().to_string())
else:
    print('无失败项。')
